In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torch
import gc

In [ ]:
# clearing GPU cache:
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
# force garbage collection:
gc.collect()
print('cache cleared and garbage collected!')

cache cleared and garbage collected!


In [ ]:
proj_path = "/content/drive/MyDrive/llm_from_scratch/src"
data_path = "/content/drive/MyDrive/llm_from_scratch/datasets"

In [ ]:
import os, sys
sys.path.append(proj_path)
sys.path.append(data_path)
os.chdir(proj_path)
print(os.getcwd())

/content/drive/MyDrive/llm_from_scratch/src


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [ ]:
from gpt_model import GPTModel
from config import CONFIG
model = GPTModel(CONFIG)

In [ ]:
model.load_state_dict(torch.load("/content/drive/MyDrive/llm_from_scratch/text_generation_model/pretrained_weights.pth"))

<All keys matched successfully>

In [ ]:
# torch.save(model.state_dict(), "/content/drive/MyDrive/llm_from_scratch/text_generation_model/pretrained_weights.pth")
# print("Pretrained weights saved for fast loading!")

In [ ]:
gc.collect()
torch.cuda.empty_cache()

In [ ]:
import tiktoken, json
from torch.utils.data import Dataset
from transformers import TrainingArguments, Trainer

In [ ]:
class JsonInstructionDataset(Dataset):
    def __init__(self, json_data, max_length=512,):
        self.encoding = tiktoken.get_encoding('gpt2')
        self.json_data = json_data
        self.max_length = max_length

    def __len__(self):
        return len(self.json_data)

    def __getitem__(self, idx):
        item = self.json_data[idx]

        text = f"### Instruction:\n{item['instruction']}\n\n### Input:\n{item['input']}\n\n### Response:\n{item['output']}"

        # tokenizing with tiktoken
        tokens = self.encoding.encode(text)

        # truncating if necessary
        if len(tokens) > self.max_length:
            tokens = tokens[:self.max_length]

        input_ids = tokens

        # padding if necessary
        if len(input_ids) < self.max_length:
            padding_length = self.max_length - len(input_ids)
            padding = [self.encoding.eot_token] * padding_length
            input_ids = input_ids + padding

        return {
            'input_ids': torch.tensor(input_ids, dtype=torch.long),
            'labels': torch.tensor(input_ids, dtype=torch.long)
        }

In [ ]:
import json

In [ ]:
with open("/content/drive/MyDrive/llm_from_scratch/datasets/alpaca_gpt4_data.json", "r", encoding="utf-8") as f:
    json_data_gpt4 = json.load(f)

In [ ]:
print(len(json_data_gpt4))

52002


In [ ]:
json_gpt4_train, json_gpt4_val = json_data_gpt4[:20000], json_data_gpt4[20000:20500]
print(len(json_gpt4_train))
print(len(json_gpt4_val))

20000
500


In [ ]:
print(json_gpt4_train[-1])
print(json_gpt4_val[-2])

{'instruction': 'Suggest a good strategy to achieve a goal.', 'input': 'Increasing revenue.', 'output': 'One good strategy to increase revenue is as follows:\n\n1. Conduct market research: Find out more about your target audience and their needs. Identify the gaps in the market where you can potentially provide value.\n\n2. Optimize pricing strategy: Evaluate your current pricing scheme and see if there is room for adjustment. Price your products or services competitively, while ensuring profitability.\n\n3. Improve customer retention: Focus on keeping your existing customers happy, as they are more likely to make repeat purchases. Implement customer loyalty programs or personalized promotions to incentivize them to make more purchases.\n\n4. Expand product or service line: Look for more ways to provide value to your target market by introducing new products or services. This could attract new customers or entice existing ones to purchase more.\n\n5. Increase marketing efforts: Increas

In [ ]:
train_dataset = JsonInstructionDataset(json_gpt4_train)
val_dataset = JsonInstructionDataset(json_gpt4_val)

In [ ]:
print(train_dataset[0])
print("successfully printed.")

{'input_ids': tensor([21017, 46486,    25,   198, 23318,  1115,  9040,   329, 10589,  5448,
           13,   198,   198, 21017, 23412,    25,   628,   198, 21017, 18261,
           25,   198,    16,    13, 27574,   257, 12974,   290, 48102,  5496,
           25,  6889,  1654,   534, 13840,   389, 19889,   286,   257,  4996,
          286, 15921,   290, 13701,    11, 10904,  7532,    11,  2187, 21824,
           11,   290,  5448, 27997,    13,   770,  5419,   284,  2148,   534,
         1767,   351,   262,  6393, 20901,   284,  2163,   379,   663,  1266,
          290,   460,  1037,  2948, 10726, 10040,    13,   198,   198,    17,
           13,  1985,   496,   287,  3218,  3518,  3842,    25, 32900,   318,
         8780,   329, 10941,  1913, 11945,    11, 12749,    11,   290, 21134,
         1535,    13, 36223,   329,   379,  1551,  6640,  2431,   286, 10768,
        43294,  5517,   393,  5441,  2431,   286, 31543,  5517,  1123,  1285,
           13,   198,   198,    18,    13,  3497, 

In [ ]:
from transformers import PretrainedConfig

class GPTConfig(PretrainedConfig):
    def __init__(self, **kwargs):
        # config
        self.vocab_size = 50257
        self.context_length = 1024
        self.emb_dim = 1280
        self.n_heads = 20
        self.n_layers = 36
        self.drop_rate = 0.1
        self.qkv_bias = True
        super().__init__(**kwargs)

# attaching config to model
model.config = GPTConfig()

In [ ]:
import types

def hf_forward(self, input_ids=None, labels=None, attention_mask=None, **kwargs):
    batch_size, seq_len = input_ids.shape

    with torch.amp.autocast('cuda', enabled=True):
        tok_embeds = self.tok_emb(input_ids)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=input_ids.device))
        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)

    loss = None
    if labels is not None:
        loss_fct = torch.nn.CrossEntropyLoss()
        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()
        loss = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))

    return (loss, logits) if loss is not None else logits

model.forward = hf_forward.__get__(model, type(model))

In [ ]:
def prepare_inputs_for_generation(self, input_ids, **kwargs):
    return {"input_ids": input_ids, **kwargs}

if not hasattr(model, 'prepare_inputs_for_generation'):
    model.prepare_inputs_for_generation = prepare_inputs_for_generation.__get__(model, type(model))

In [ ]:
from peft import LoraConfig, get_peft_model

# model = model.half()

def setup_lora_model(model):

    target_modules = [
        "W_query", "W_key", "W_value",
        "out_proj",
        "out_head"
    ]

    # LoRA configuration
    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=target_modules,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        modules_to_save=None
    )

    # applying LoRA
    model = get_peft_model(model, lora_config)

    return model

# applying LoRA to model
model = setup_lora_model(model)

In [ ]:
import transformers

transformers.logging.set_verbosity_info()

training_args = TrainingArguments(

    output_dir="/content/drive/MyDrive/llm_from_scratch/text_generation_model/instruction_model",

    num_train_epochs=1,
    learning_rate=3e-5,

    # Batch size and gradient settings
    per_device_train_batch_size=2,
    # per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,

    # Optimization settings
    optim="adamw_torch",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,

    # Mixed precision training
    fp16=True,

    # Evaluation settings
    # eval_strategy="steps",
    # eval_steps=30,

    # Data loading settings
    dataloader_pin_memory=False,
    dataloader_drop_last=True,
    remove_unused_columns=False,

    # Progress and logging
    logging_steps=10,
    disable_tqdm=False,
    report_to="none",

    # Save settings
    save_strategy="steps",
    save_steps=250,
    save_total_limit=1
)

PyTorch: setting up devices


In [ ]:
def compute_metrics(eval_pred):
    return {}

In [ ]:
os.environ["WANDB_DISABLED"] = "true"
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset
    # eval_dataset=val_dataset,
    # compute_metrics=compute_metrics
)

Using auto half precision backend


In [ ]:
gc.collect()
torch.cuda.empty_cache

print("Started training...")
trainer.train()
print("Training completed!")

***** Running training *****
  Num examples = 20,000
  Num Epochs = 1
  Instantaneous batch size per device = 2
  Total train batch size (w. parallel, distributed & accumulation) = 16
  Gradient Accumulation steps = 8
  Total optimization steps = 1,250
  Number of trainable parameters = 3,361,416


Started training...


Step,Training Loss
10,55.177400
20,57.544700
30,53.676300
40,55.368000
50,50.026300
60,43.390200
70,35.757500
80,24.128500
90,15.076900
100,11.375700


Saving model checkpoint to /content/drive/MyDrive/llm_from_scratch/text_generation_model/instruction_model/checkpoint-250
Saving model checkpoint to /content/drive/MyDrive/llm_from_scratch/text_generation_model/instruction_model/checkpoint-500
Deleting older checkpoint [/content/drive/MyDrive/llm_from_scratch/text_generation_model/instruction_model/checkpoint-250] due to args.save_total_limit
Saving model checkpoint to /content/drive/MyDrive/llm_from_scratch/text_generation_model/instruction_model/checkpoint-750
Deleting older checkpoint [/content/drive/MyDrive/llm_from_scratch/text_generation_model/instruction_model/checkpoint-500] due to args.save_total_limit


Step,Training Loss
10,55.177400
20,57.544700
30,53.676300
40,55.368000
50,50.026300
60,43.390200
70,35.757500
80,24.128500
90,15.076900
100,11.375700


In [ ]:
# def test_model(model, instruction, input_text="", max_new_tokens=1024):
#     # Format the prompt like during training
#     if input_text:
#         prompt = f"### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n"
#     else:
#         prompt = f"### Instruction:\n{instruction}\n\n### Response:\n"

#     # Tokenize
#     encoding = tiktoken.get_encoding('gpt2')
#     input_ids = encoding.encode(prompt)
#     input_tensor = torch.tensor([input_ids]).to(device)

#     # Manual generation (no .generate() method)
#     model.eval()
#     with torch.no_grad():
#         generated = input_tensor

#         for i in range(max_new_tokens):
#             # Forward pass
#             # Access logits directly from the returned tensor
#             logits = model(input_ids=generated)

#             # Get last token logits
#             next_token_logits = logits[0, -1, :]

#             # Apply temperature and sample
#             next_token_logits = next_token_logits / 0.7  # temperature
#             probs = torch.softmax(next_token_logits, dim=-1)
#             next_token = torch.multinomial(probs, num_samples=1)

#             # Stop if EOT token is generated BEFORE appending
#             if next_token.item() == encoding.eot_token:
#                 break

#             # Append to generated sequence
#             generated = torch.cat([generated, next_token.unsqueeze(0)], dim=1)

#     # Decode
#     response = encoding.decode(generated[0].tolist())
#     # Extract only the response part (after "### Response:\n")
#     response_text = response.split("### Response:\n")[-1]

#     # Remove endoftext token if it exists and clean up
#     response_text = response_text.replace('<|endoftext|>', '').strip()

#     return response_text

# # Test examples
# print("🧪 Testing the fine-tuned model:\n")

# # Example 1: General instruction
# test1 = test_model(
#     model,
#     instruction="how can i learn english faste explain in points.",
#     input_text="",
# )
# print(f"Test 1 - Explanation:\n{test1}\n")

In [ ]:
# After instruction fine-tuning, merge LoRA into base model
merged_model = model.merge_and_unload()

# Save the MERGED weights (now base model has instruction knowledge)
torch.save(merged_model.state_dict(), "/content/drive/MyDrive/llm_from_scratch/text_generation_model/instruction_model/instruction_finetuned_model.pth")

print("✅ Saved unified model with instruction knowledge")